# Video to Gaussian scene
Private T4 notebook. Enable Internet and GPU T4. Choose VIDEO_PATH below for standalone use.
The worker uses one T4; the second T4 is unused. Results remain under /kaggle/working/run and bridge jobs upload validated artifacts.


In [ ]:
# Edit VIDEO_PATH for a standalone uploaded Kaggle dataset.
import json
CONFIG = json.loads('{"payload": null, "video": "Video_0.mp4", "output": "/kaggle/working/run", "pose_estimator": "glomap", "pipeline_args": [], "bridge_smoke": false}')
VIDEO_PATH = CONFIG['video']


In [ ]:
from pathlib import Path
import base64, io, os, sys, tarfile
_archive = '...'
SOURCE_ROOT = Path('/kaggle/working/reconstruction-source')
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(_archive)), mode='r:gz') as bundle:
    for member in bundle.getmembers():
        target = (SOURCE_ROOT / member.name).resolve()
        if not target.is_relative_to(SOURCE_ROOT.resolve()) or not member.isfile():
            raise RuntimeError('Invalid packaged source path')
    bundle.extractall(SOURCE_ROOT, filter='data')
del _archive
os.chdir(SOURCE_ROOT)
sys.path.insert(0, str(SOURCE_ROOT))
from reconstruction.kaggle.worker import BridgeClient, ProgressPump, run
from reconstruction.kaggle.setup_environment import prepare, VGGT_REVISION
from contextlib import nullcontext
client = BridgeClient(CONFIG['payload']) if CONFIG['payload'] else None
try:
    if CONFIG.get('bridge_smoke'):
        if client is None: raise RuntimeError('Bridge smoke needs a scoped payload')
        smoke_dir = Path(CONFIG['output'])
        smoke_dir.mkdir(parents=True, exist_ok=True)
        client.progress({'stage':'testing_bridge','progress':None,'message':'Kaggle HTTPS callback reached bridge'})
        downloaded = client.download(smoke_dir / 'input.mp4')
        client.progress({'stage':'bridge_smoke_complete','progress':None,'message':f'Input downloaded: {downloaded.stat().st_size} bytes; no GPU reconstruction requested'})
    else:
        if client is None and not Path(VIDEO_PATH).is_file():
            matches = list(Path('/kaggle/input').rglob(VIDEO_PATH)) if Path(VIDEO_PATH).name == VIDEO_PATH else []
            if len(matches) != 1:
                raise ValueError('Set VIDEO_PATH to the exact path of one attached video in the configuration cell')
            VIDEO_PATH = str(matches[0])
        with (ProgressPump(client) if client else nullcontext()) as pump:
            preparation = {'event':'progress','stage':'preparing_environment','progress':None,
                           'message':'Preparing the pinned CUDA and reconstruction environment'}
            print(json.dumps(preparation), flush=True)
            if pump: pump.emit(preparation)
            runtime = prepare(with_vggt=CONFIG['pose_estimator']=='vggt' or '--compare-poses' in CONFIG['pipeline_args'], output='/kaggle/working/environment-report')
        pipeline_args = list(CONFIG['pipeline_args'])
        if (CONFIG['pose_estimator']=='vggt' or '--compare-poses' in pipeline_args) and '--vggt-code-revision' not in pipeline_args:
            pipeline_args += ['--vggt-code-revision', VGGT_REVISION]
        run(payload=CONFIG['payload'], video=VIDEO_PATH, output=CONFIG['output'],
            pose_estimator=CONFIG['pose_estimator'], pipeline_args=pipeline_args, **runtime)
except Exception as error:
    if client:
        safe = str(error).replace(CONFIG['payload']['token'], '[REDACTED]')
        try: client.progress({'event':'failed','stage':'failed','message':safe,'error':safe})
        except Exception: print('Could not deliver failure callback; inspect private kernel status')
    raise
